# Phase 2: Feedback stage

This notebook produces feedback on first codes and then justifications in two seperate steps

### Read draft output DF

In [ ]:
import pandas as pd
import json

draft_df = pd.read_parquet(DRAFT_DIR / "currentTEST.parquet")

draft_df.columns


### Exploding draft_df to evaluate codes individually - making new feedback df

In [ ]:
#Convert arrays to lists
draft_df["codes"] = draft_df["codes"].apply(lambda x: x.tolist())
draft_df["justifications"] = draft_df["justifications"].apply(lambda x: x.tolist())

In [ ]:
#Make new feedback df

feedback_df = draft_df.copy()

#Explode df
feedback_df = feedback_df.explode(
    ["codes", "justifications"],
    ignore_index=True
)

In [ ]:
#Make code_UIDs for anchoring

feedback_df["code_id"] = (
    feedback_df.groupby("unit_id").cumcount() + 1
)

feedback_df["code_uid"] = (
    feedback_df["int_id"].astype(str) + "_" +
    feedback_df["unit_id"].astype(str) + "_" +
    feedback_df["code_id"].astype(str)
)

In [ ]:
#Make grouped DF - analytical object to pass to function

grouped_df = feedback_df.groupby("unit_id").agg({
    "int_id": "first",
    "interview_excerpt": "first",
    "codes": list,
    "justifications": list,
    "code_id": list,
    "code_uid": list
}).reset_index()

### Load LLM model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id) #Mistral's tokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" #A100 or H100
)

print("Mistral-7B-Instruct-v0.3 is loaded.")

### LLM configurations w. Outlines

In [ ]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_model_config(model.config)

model.generation_config.max_new_tokens = 400 #max token outout
model.generation_config.min_new_tokens = 40 #min new tokens
model.generation_config.temperature = 0.0 # Temp set to 0.0 because it is overridden by Outlines in structured output
model.generation_config.do_sample = False #LLM chooses most likely next token = False
model.generation_config.pad_token_id = tokenizer.eos_token_id # Mistral does not have a pad token, so we pad with EOS
model.generation_config.eos_token_id = tokenizer.eos_token_id #Mistral's end-of-sequence token

### Import Outlines for structured output

In [ ]:
import outlines

outlines_model = outlines.from_transformers(model, tokenizer)

## Analyses prompt - feedback stage CODE EVAL

- We have two prompts with two seperate tasks
- In this first prompt we focus only on evaulating codes

### Format draft stage prompt in INST (chat_template)

In [ ]:
def format_prompt(content, tokenizer): #Taking analysis prompt and sending it to model in instrct mode
    messages = [
        {"role": "user", "content": content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

### Prompt for feedback on codes

In [ ]:
# Feedback stage function prompt V1.4 CODES

def get_feedback_stage(interview_excerpt, codes):
    content = f"""

    ### TASK
    You are an expert qualitative researcher. You are evaluating qualitative coding (Braun & Clarke 2006)
    For each code, assess the quality of the connection between the interview excerpt and the code, including:
    1) how clearly the code is grounded in the excerpt
    2) how relevant the code is to the research objective

    ### CONTEXT
    The research objective is to understand the interviewees’ individual experiences of the profound transformations in Danish society
    and family life from the 1960s-1980s, characterized by changing gender roles, women’s increasing labour market participation,
    the weakening of traditional marriage, and the expansion of family policies.

    ###CRITERIA
    For each code, you must assign exactly one label AND provide a justification.


    ### EVALUATION REQUIREMENTS

    #### Label definitions:

    Mark [good] if:
    - The code is grounded in the excerpt, even if the link is implicit
    - OR the code captures a plausible aspect of the interviewee’s experience relevant to the research objective

    Mark [bad] if:
    - The connection between excerpt and code is weak, unclear, or only loosely supported
    - OR the relevance to the research objective is uncertain but not absent

    Mark [not_relevant] ONLY if:
    - The excerpt contains purely descriptive or background information (e.g. age, factual details)
    - AND there is no meaningful experiential or interpretive content related to the research objective

    When in doubt between [bad] and [not_relevant], choose [bad].

    #### Justification requirements:
    Each justification must:
    - be 1–2 sentences
    - directly explain why the label was assigned
    - be grounded in the interview_excerpt
    - not be generic (e.g. avoid vague statements like "this seems relevant")
    - not repeat the code name

    ###CODES TO EVALUATE
    Codes:
    {codes}

    Interview_excerpt:
    {interview_excerpt}


    ### INSTRUCTIONS FOR RESPONSE
   -  Return exactly one evaluation per code, in the same order.
    - Each evaluation must be one of: good, bad, not_relevant.
    - Do not include placeholders such as "none".
    - Do not include any explanatory text outside the structured output.


    ### FINAL INSTRUCTION
    Evaluate the codes for each interview_excerpt.
    """
    return content



## Defining output schema for code eval with Pydantic

In [ ]:
#Def Schema for output
from pydantic import BaseModel, Field
from typing import List, Literal



def make_code_eval_schema(n_codes):
    class CodeEvaluation(BaseModel):
        label: Literal["good", "bad", "not_relevant"]
        justification: str = Field(
            description="Short reasoning for the assigned label",
            min_length=1,
            max_length=300,
        )

    class CodeEvaluationResponse(BaseModel):
        evaluations: List[CodeEvaluation] = Field(
            min_length=n_codes,
            max_length=n_codes,
            description=f"Exactly {n_codes} evaluations required"
        )

    return CodeEvaluationResponse

# Main function for evaluating codes in feedback stage, parsing and validating with Outlines

In [ ]:
import json
import time

def evaluate_codes(interview_excerpt, codes, max_retries=3, retry_delay=0.1):
    content = get_feedback_stage(interview_excerpt, codes)
    prompt_inst = format_prompt(content, tokenizer)

    # Dynamic lenght (n_codes must match n_evals)
    n_codes = len(codes)
    schema = make_code_eval_schema(n_codes)

    last_error = None

    for attempt in range(max_retries):
        try:
            result = outlines_model(prompt_inst, schema)

            # Case 1: if output is already correct
            if isinstance(result, schema):
                validated = result

            # Case 2: if string, then parse
            elif isinstance(result, str):
                parsed = json.loads(result)

                if isinstance(parsed, list):
                    parsed = {
                        "evaluations": [
                            {"label": item} for item in parsed
                        ]
                    }

                validated = schema(**parsed) #then put on results list

            else:
                raise ValueError(f"Unexpected result type: {type(result)}")

            # Deterministic check (aligment with Pydantic schema)
            if len(validated.evaluations) != n_codes:
                raise ValueError(
                    f"Length mismatch: {len(validated.evaluations)} vs {n_codes}"
                )

            return validated

        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    raise ValueError(f"Failed after {max_retries} attempts: {last_error}")

## Function for processing each row and picking up errors - loop

In [ ]:

def process_row(row):
    print("NEW VERSION RUNNING")
    try:
        validated = evaluate_codes(
            row["interview_excerpt"],
            row["codes"]
        )

        print(
            f"codes={len(row['codes'])}, "
            f"uids={len(row['code_uid'])}, "
            f"evals={len(validated.evaluations)}"
        )

        rows = []

        for code_uid, code, item in zip(
            row["code_uid"],
            row["codes"],
            validated.evaluations
        ):
            rows.append({
                "unit_id": int(row["unit_id"]),
                "int_id": row["int_id"], #er det her?
                "code_uid": code_uid,
                "code": code,
                "code_label": item.label,
                "code_label_justification": item.justification,
                "status": "ok",
                "error": None,
                "excerpt": row["interview_excerpt"]
            })

        return rows

    except Exception as e:
        return [{
            "unit_id": int(row["unit_id"]),
            "code_uid": None,
            "int_id": row["int_id"], #og her?
            "code": None,
            "code_label": None,
            "code_label_justification": None,
            "status": "failed",
            "error": str(e),
            "excerpt": row["interview_excerpt"]
        }]

### Run pipeline - call functions



In [ ]:
import time
import pandas as pd

unique_int_ids = grouped_df["int_id"].unique()
total_interviews = len(unique_int_ids)

print(f"Starting feedback phase... ({total_interviews} interviews)\n", flush=True)

start_total = time.time()
results = []

for i, int_id in enumerate(unique_int_ids, start=1):
    interview_start = time.time()

    print(f"\n--- Interview {i}/{total_interviews}: {int_id} ---", flush=True)

    df_subset = grouped_df[grouped_df["int_id"] == int_id]

    print(f"Number of excerpts: {len(df_subset)}", flush=True)

    for j, (_, row) in enumerate(df_subset.iterrows(), start=1):
        row_start = time.time()

        print(f"Processing row {j}/{len(df_subset)}", flush=True)

        try:
            result = process_row(row)
            #results.append(result)
            results.extend(result)

            row_time = time.time() - row_start
            print(f"Finished row {j} in {round(row_time, 2)} sec", flush=True)

        except Exception as e:
            print(f"Error on row {j}: {e}", flush=True)
            continue

    interview_time = time.time() - interview_start
    print(f"Finished interview {int_id} in {round(interview_time, 2)} sec", flush=True)

total_time = time.time() - start_total

results_df = pd.DataFrame(results)

print(f"\nTotal time: {round(total_time, 2)} sec ({round(total_time/60, 2)} min)", flush=True)

### Save results from evaluation of codes

In [ ]:
final_feedback_df = feedback_df.merge(
    results_df[["code_uid", "code_label", "code_label_justification"]],
    on="code_uid",
    how="left"
)


In [ ]:
final_feedback_df.columns

Index(['unit_id', 'int_id', 'interview_excerpt', 'segment_id', 'codes',
       'justifications', 'status', 'error', 'code_id', 'code_uid',
       'code_label', 'code_label_justification'],
      dtype='object')

In [ ]:
final_feedback_df["code_label"].isna().sum()

np.int64(0)

# Save as Parquet

In [ ]:
from datetime import datetime
import shutil
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# PARQUET (source of truth for next step in self-refine flow)
timestamp_path_parquet = FEEDBACK_DIR / f"feedback_{timestamp}.parquet"
current_path_parquet = FEEDBACK_DIR / "currentTEST.parquet"

final_feedback_df.to_parquet(timestamp_path_parquet, index=False)
shutil.copy(timestamp_path_parquet, current_path_parquet)



## STEP 2: Evaluation of justifications
- only for codes that are labeled good or bad
- not_relevant codes are removed

### Read output from feedback stage to get evaluation of codes

In [ ]:
import pandas as pd
import json

feedback_df_2 = pd.read_parquet(FEEDBACK_DIR / "currentTEST.parquet")

feedback_df_2.columns

### New sorted DF for evaluation of justifications

In [ ]:
# Keep code_UID as unique ID and remove codes that are assessed not_relevant

sorted_df = feedback_df_2[feedback_df_2['code_label'].isin(['good', 'bad'])].copy()

print(f"Rows for eval: {len(sorted_df)}")
print(f"Rows excluded (not_relevant): {len(feedback_df) - len(sorted_df)}")

## Analysis prompt - for evaluation of justifications

In [ ]:
# Feedback stage function prompt JUSTIFICATIONS

def get_justification_feedback_stage(interview_excerpt, codes, justifications):
    content = f"""

    ### TASK
    You are an expert qualitative researcher specializing in thematic analysis (Braun & Clarke 2006).
    Your task is to evaluate if the following justification for a code is analytically weak.
    Specifically, identify if it is "vague" or "paraphrastic".

    ### DATA
    - **Interview Excerpt:** {interview_excerpt}
    - **Code:** {codes}
    - **Justification to Evaluate:** {justifications}

    ### EVALUATION CRITERIA
    Assign exactly one label:
    1. [vague_or_paraphrastic]: If the justification is just a paraphrase (repeating the text) or is too generic/vague.
    2. [good]: If the justification provides a clear and specific analytical link.

    ### INSTRUCTIONS FOR RESPONSE
    - Provide the label for 'justification_label': [good] or [vague_or_paraphrastic].
    - Provide a 1-sentence explanation for 'justification_eval_explanation' explaining specifically why the label was assigned.
    - Do not include any explanatory text outside the structured output.
    """

    return content


## Updated schema for structured output (justifications)

In [ ]:
# Def Schema for justification output
from pydantic import BaseModel, Field
from typing import List, Literal

def make_justification_eval_schema(n_justifications):
    class JustificationEvaluation(BaseModel):
        # Categorical label
        justification_label: Literal["good", "vague_or_paraphrastic"] = Field(
            description="Label for the justification quality"
        )
        # Exlplanation for label
        justification_eval_explanation: str = Field(
            description="1-sentence critique explaining specifically why the label was assigned",
            min_length=1,
            max_length=400,
        )

    class JustificationEvaluationResponse(BaseModel):
        evaluations: List[JustificationEvaluation] = Field(
            min_length=n_justifications,
            max_length=n_justifications,
            description=f"Exactly {n_justifications} evaluations required"
        )

    return JustificationEvaluationResponse

### Updated eval function

In [ ]:
import json
import time

def evaluate_justifications(interview_excerpt, code, justifications, max_retries=3, retry_delay=0.1):

    content = get_justification_feedback_stage(interview_excerpt, code, justifications)
    prompt_inst = format_prompt(content, tokenizer)

    # align output format to list to align with schema definitions
    if isinstance(justifications, str):
        justifications = [justifications]

    n_justs = len(justifications)

    # 3. Apply updated justification schema
    schema = make_justification_eval_schema(n_justs)

    last_error = None

    for attempt in range(max_retries):
        try:
            result = outlines_model(prompt_inst, schema)


            # Case 1: if output is already correct (Pydantic object)
            if isinstance(result, schema):
                validated = result

            # Case 2: if string, then parse
            elif isinstance(result, str):
                parsed = json.loads(result)

                # Opdated logic
                if isinstance(parsed, list):
                    parsed = {
                        "evaluations": [
                            {
                                "justification_label": item.get("justification_label") if isinstance(item, dict) else item,
                                "justification_eval_explanation": item.get("justification_eval_explanation") if isinstance(item, dict) else ""
                            } for item in parsed
                        ]
                    }

                validated = schema(**parsed)

            else:
                raise ValueError(f"Unexpected result type: {type(result)}")

            # 4. Check alignment w/ no of justifications
            if len(validated.evaluations) != n_justs:
                raise ValueError(
                    f"Length mismatch: {len(validated.evaluations)} vs {n_justs}"
                )

            return validated


        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    raise ValueError(f"Failed after {max_retries} attempts: {last_error}")


## Updated process row function

In [ ]:
def process_justification_row(row):
    try:

        validated = evaluate_justifications(
            interview_excerpt=row["interview_excerpt"],
            code=row["codes"],
            justifications=[row["justifications"]] # Send as list to match the schema logic
        )

        # take the eval_item from Pydantic
        eval_item = validated.evaluations[0]

        return {
            "code_uid": row["code_uid"], # Match-key for DF
            "justification_label": eval_item.justification_label,
            "justification_eval_explanation": eval_item.justification_eval_explanation,
            "status": "ok"
        }

    except Exception as e:
        return {
            "code_uid": row["code_uid"],
            "justification_label": "error",
            "justification_eval_explanation": str(e),
            "status": "failed"
        }

In [ ]:
import time
import pandas as pd

# Use sorted_df
unique_int_ids = sorted_df["int_id"].unique()
total_interviews = len(unique_int_ids)

print(f"Starting JUSTIFICATION feedback phase... ({total_interviews} interviews)\n", flush=True)

start_total = time.time()
just_results = []

for i, int_id in enumerate(unique_int_ids, start=1):
    interview_start = time.time()

    print(f"\n--- Interview {i}/{total_interviews}: {int_id} ---", flush=True)

    # inner loop per interview for analytical consistency
    df_subset = sorted_df[sorted_df["int_id"] == int_id]

    print(f"Number of justifications to evaluate: {len(df_subset)}", flush=True)

    for j, (_, row) in enumerate(df_subset.iterrows(), start=1):
        row_start = time.time()

        # Print code_uid
        print(f"Processing justification {j}/{len(df_subset)} (UID: {row['code_uid']})", flush=True)

        try:
            # calling new function for eval of justifications
            result = process_justification_row(row)
            just_results.append(result)

            row_time = time.time() - row_start
            print(f"Finished row {j} in {round(row_time, 2)} sec", flush=True)

        except Exception as e:
            print(f"Error on row {j} (UID: {row['code_uid']}): {e}", flush=True)
            continue

    interview_time = time.time() - interview_start
    print(f"Finished interview {int_id} in {round(interview_time, 2)} sec", flush=True)

total_time = time.time() - start_total

# Collect results to new DF
just_results_df = pd.DataFrame(just_results)



print(f"\nTotal time: {round(total_time, 2)} sec ({round(total_time/60, 2)} min)", flush=True)


In [ ]:
#  Merge resultats back to original current DF (NOT sorted_df) for full traceability - paste the results from justification_eval back to codes_eval

final_df = feedback_df_2.merge(
    just_results_df[['code_uid', 'justification_label', 'justification_eval_explanation']],
    on='code_uid',
    how='left'
)
print(f"Justification feedback complete. Results merged into final_df.", flush=True)

### Save as parquet

In [ ]:
from datetime import datetime
import shutil
import json


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# PARQUET (Source of truth for refine step) ---
timestamp_path_parquet = FEEDBACK_DIR / f"justification_feedback_{timestamp}.parquet"
current_path_parquet = FEEDBACK_DIR / "current_feedback_final_TEST.parquet"

final_df.to_parquet(timestamp_path_parquet, index=False)
shutil.copy(timestamp_path_parquet, current_path_parquet)



Files are saved in: /work/NadiaJulJeldtoft#6373/Speciale/Results_model_coding/Exp1.1/feedback
